# 第5章 モデルに点数をつける：損失とパープレキシティ

確率を掛けたら0になった事件と、対数という救世主

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/05-loss.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## 準備：『こころ』を読み込む


In [ ]:
import re, math, urllib.request
from collections import Counter

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/master/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    text = urllib.request.urlopen(url).read().decode("cp932")
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
print(len(text), "文字")

## 過去問と本番：データを分ける


In [ ]:
n = int(len(text) * 0.9)
train_text = text[:n]   # 前の90%：学習用（数える用）
val_text = text[n:]     # 後ろの10%：点数をつける用
print("訓練データ：", len(train_text), "文字")
print("検証データ：", len(val_text), "文字")

In [ ]:
pairs = Counter(zip(train_text, train_text[1:]))
firsts = Counter(train_text[:-1])

def prob(a, b):
    """P(b | a)：「a」の次が「b」である確率"""
    if firsts[a] == 0:
        return 0.0
    return pairs[(a, b)] / firsts[a]

print("P(生|先) =", prob("先", "生"))
print("P(は|生) =", prob("生", "は"))

## 文章の確率を計算する


In [ ]:
p = 1.0
for i, (a, b) in enumerate(zip(train_text, train_text[1:])):
    p = p * prob(a, b)
    if i in [10, 50, 100, 150, 200, 300]:
        print(f"{i:4d}文字目まで：確率 {p}")

## 救世主：対数（log）


In [ ]:
print(math.log10(1000), math.log10(0.01))
print(math.log(math.e), math.log(1))
print(math.log(0.5), math.log(0.01))

In [ ]:
log_p = 0.0
for i, (a, b) in enumerate(zip(train_text, train_text[1:])):
    log_p = log_p + math.log(prob(a, b))
    if i in [10, 50, 100, 200, 300]:
        print(f"{i:4d}文字目まで：log確率 {log_p:.1f}")

## 損失：驚きの平均


In [ ]:
!pip install -q matplotlib-fontja

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

ps = [i / 100 for i in range(1, 101)]
plt.figure(figsize=(6, 3.5))
plt.plot(ps, [-math.log(p) for p in ps])
plt.xlabel("モデルが予想した確率 p")
plt.ylabel("驚き  -log p")
plt.title("確率が低いことが起きるほど、驚きは大きい")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
def bigram_loss(data, prob_fn):
    total = 0.0
    for a, b in zip(data, data[1:]):
        p = prob_fn(a, b)
        if p == 0:
            return float("inf")       # 確率0の出来事 → 驚きは無限大
        total += -math.log(p)
    return total / (len(data) - 1)

print("訓練データの損失：", bigram_loss(train_text, prob))
print("検証データの損失：", bigram_loss(val_text, prob))

In [ ]:
unseen = sum(1 for a, b in zip(val_text, val_text[1:]) if pairs[(a, b)] == 0)
print("検証データのペア：", len(val_text) - 1, "個のうち、訓練で見ていないもの：", unseen, "個")

## 「絶対」を言わないモデル：スムージング


In [ ]:
V = len(set(text))

def make_smoothed(k):
    def p(a, b):
        return (pairs[(a, b)] + k) / (firsts[a] + k * V)
    return p

for k in [1, 0.1, 0.01, 0.001]:
    p_k = make_smoothed(k)
    print(f"k={k:<6} 訓練 {bigram_loss(train_text, p_k):.3f}   検証 {bigram_loss(val_text, p_k):.3f}")

## パープレキシティ：何択で迷っているか


In [ ]:
print("bigram（k=0.01）のパープレキシティ：", math.exp(3.581))
print("でたらめに選ぶモデルの損失：", math.log(V), " パープレキシティ：", math.exp(math.log(V)))